# SmartCart Day 4 - Augment the Long Tail

Day 3's error report named the weak SKUs - usually the rare ones with too few examples. Today we first use **classical augmentation** as the reliable baseline, then leave a **cGAN/equivalent generative option** for teams that want to go further. Either way, the final judge is weak-class validation lift.

In [1]:
# 1) Runtime setup
# This course runs in Google Colab. Run this cell first.
# Install only the packages used in this notebook.
%pip install -q timm

import os

# Drive stores the small cross-day bundle, not the image dataset.
from google.colab import drive
drive.mount('/content/drive')
BUNDLE_DIR = '/content/drive/MyDrive/snaic/week_4/Project/SmartCart'



Mounted at /content/drive


### Embedded toolkit

This cell defines the helper functions used below. Run it once after setup.

In [2]:
from __future__ import annotations
import json
import os
import pathlib
import subprocess
import numpy as np
import pandas as pd
HERE = pathlib.Path.cwd()  # embedded in-notebook: no __file__, anchor on the working dir
GROCERY_DATASET_URL = 'https://github.com/marcusklasson/GroceryStoreDataset'

class Bundle:
    """Small Google Drive folder that carries artifacts from one day to the next."""

    def __init__(self, root: str):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.manifest = {'version': 1, 'class_list': [], 'artifacts': {}}

    def put_table(self, name, df: pd.DataFrame):
        df.to_csv(self.root / name, index=False)
        self._note(name)

    def get_table(self, name) -> pd.DataFrame:
        return pd.read_csv(self.root / name)

    def put_array(self, name, arr: np.ndarray):
        np.save(self.root / name, arr)
        self._note(name)

    def get_array(self, name) -> np.ndarray:
        p = self.root / name
        return np.load(p if p.suffix == '.npy' else p.with_suffix('.npy'))

    def _note(self, name):
        self.manifest['artifacts'][name] = True

    def save(self):
        (self.root / 'manifest.json').write_text(json.dumps(self.manifest, indent=2))

    def load(self):
        p = self.root / 'manifest.json'
        if p.exists():
            self.manifest = json.loads(p.read_text())
        return self

def open_bundle(drive_dir='/content/drive/MyDrive/SmartCart') -> Bundle:
    """Open the cross-day Drive bundle. If it is new, start with an empty manifest."""
    return Bundle(drive_dir).load()

def save_bundle(b: Bundle):
    b.save()
    print(f'[bundle] saved -> {b.root}')

def open_grocery_dataset():
    """Clone or reuse the real GroceryStoreDataset and return (tier, root_dir)."""
    root = HERE / 'GroceryStoreDataset'
    if (root / 'dataset' / 'classes.csv').exists():
        print('using cached GroceryStoreDataset:', root)
    else:
        print('cloning GroceryStoreDataset from:', GROCERY_DATASET_URL)
        subprocess.run(['git', 'clone', '--depth', '1', GROCERY_DATASET_URL, str(root)], check=True)
    if not (root / 'dataset' / 'classes.csv').exists():
        raise RuntimeError('GroceryStoreDataset clone is incomplete. Check Colab network access and rerun.')
    print('using data tier: github')
    print('data root:', root)
    print('OK: using real GroceryStoreDataset images.')
    return ('github', str(root))

def list_images(root, per_class=None):
    """Return a DataFrame with columns path,coarse,fine from GroceryStoreDataset train/val/test folders."""
    root = pathlib.Path(root)
    rows = []
    for split in ('train', 'val', 'test'):
        for p in (root / 'dataset' / split).rglob('*.jpg'):
            rows.append({'path': str(p), 'coarse': p.parent.parent.name, 'fine': p.parent.name})
    df = pd.DataFrame(rows)
    if per_class and len(df):
        df = df.groupby('fine', group_keys=False).head(per_class).reset_index(drop=True)
    return df

def load_backbone(name='vit_small_patch14_dinov2.lvd142m', device='cpu'):
    """Frozen DINOv2-small feature extractor."""
    import timm
    m = timm.create_model(name, pretrained=True, num_classes=0, dynamic_img_size=True).eval().to(device)
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def extract_features(model, batches, device=None) -> np.ndarray:
    """Run image batches through a frozen backbone and return numpy features."""
    import torch
    if device is None:
        try:
            device = next(model.parameters()).device
        except (AttributeError, StopIteration):
            device = 'cpu'
    outs = []
    with torch.no_grad():
        for xb in batches:
            y = model(xb.to(device))
            outs.append(y.detach().cpu().numpy().astype('float32'))
    return np.concatenate(outs, 0)

class LinearHead:
    """Tiny torch linear classifier trained on frozen image features."""

    def __init__(self, in_dim, n_classes):
        import torch
        import torch.nn as nn
        self.net = nn.Linear(in_dim, n_classes)
        self.torch = torch

    def fit(self, X, y, epochs=30, lr=0.01):
        t = self.torch
        opt = t.optim.Adam(self.net.parameters(), lr=lr)
        lossf = t.nn.CrossEntropyLoss()
        Xt = t.tensor(X)
        yt = t.tensor(y)
        for _ in range(epochs):
            opt.zero_grad()
            loss = lossf(self.net(Xt), yt)
            loss.backward()
            opt.step()
        return self

    def predict(self, X):
        with self.torch.no_grad():
            return self.net(self.torch.tensor(X)).argmax(1).numpy()

def save_head(head: LinearHead, path):
    import torch
    torch.save({'state_dict': head.net.state_dict(), 'in_dim': head.net.in_features, 'n_classes': head.net.out_features}, path)
    return path
# --- helpers are now available as plain functions/classes in this notebook ---
print('SmartCart toolkit ready')


SmartCart toolkit ready


In [3]:
# 2) Load the cross-day bundle
# The bundle stores artifacts we create during the week: labels, indexes, weights, ONNX files.
# It is NOT the image dataset. Images are loaded in the next data-source cell.
b = open_bundle(BUNDLE_DIR)
print('bundle:', b.root)
print('artifacts:', list(b.manifest.get('artifacts', {})))


bundle: /content/drive/MyDrive/snaic/week_4/Project/SmartCart
artifacts: ['gallery_index.npy', 'gallery_meta.csv', 'catalog_prices.csv', 'labels.csv', 'sample_scene.jpg', 'detector.pt', 'crops_manifest.csv', 'head.pt', 'per_class_metrics.csv', 'error_report.md']


In [4]:
# 3) Select the image data source
# For class runs we require the real GroceryStoreDataset. If GitHub is unavailable, stop clearly.
tier, root = open_grocery_dataset()


cloning GroceryStoreDataset from: https://github.com/marcusklasson/GroceryStoreDataset
using data tier: github
data root: /content/GroceryStoreDataset
OK: using real GroceryStoreDataset images.


## Find rare SKUs

**What:** Load `per_class_metrics.csv` and pick the lowest-accuracy / lowest-support classes.

**Why:** We spend our augmentation budget where the model is weakest, not uniformly.

**Watch for:** If everything is already high-acc, we still demo the pipeline on the bottom few.

In [5]:
import pandas as pd
pcm = b.get_table('per_class_metrics.csv')
# Pick the weakest classes first: low accuracy, then low support.
rare = pcm.sort_values(['acc','n']).head(3).fine.tolist()
print('targeting rare/weak classes:', rare)


targeting rare/weak classes: ['Alpro-Shelf-Soy-Milk', 'Alpro-Blueberry-Soyghurt', 'Yoggi-Strawberry-Yoghurt']


## Augment weak classes

**What:** Apply heavy photometric + geometric augmentation to existing crops of the rare classes.

**Why:** More varied views of a rare SKU teach the head invariances it couldn't learn from a few photos.

**Watch for:** An optional generative model (the Day-4 stack) could replace this - we keep a classical default so CI needs no extra deps.

In [6]:
import os, torch, torchvision.transforms.v2 as T
from PIL import Image
import numpy as np
# Start from real GroceryStoreDataset images re-listed in this runtime.
src = list_images(root)
# Classical augmentations: cheap, explainable, and deterministic enough for a lab.
aug = T.Compose([T.RandomHorizontalFlip(1.0), T.RandomRotation(25),
                 T.ColorJitter(0.4,0.4,0.4,0.1), T.RandomResizedCrop(140, scale=(0.7,1.0))])
os.makedirs('augmented', exist_ok=True)
aug_rows = []
for fine in rare:
    paths = src[src.fine==fine].path.tolist()
    if not paths:
        continue
    for k in range(8):
        p = paths[k % len(paths)]
        im = T.ToImage()(Image.open(p).convert('RGB'))
        sp = f'augmented/{fine}_{k}.jpg'
        T.ToPILImage()(aug(im)).save(sp)
        aug_rows.append({'path': sp, 'fine': fine})
aug_df = pd.DataFrame(aug_rows)
print('created', len(aug_df), 'augmented images for', len(rare), 'classes')


created 24 augmented images for 3 classes


## Optional: cGAN/equivalent generator

**What:** Advanced teams may generate extra rare-class images with a small conditional GAN or an equivalent generative model.

**Why:** This connects the Day-4 generative-AI material to the smart-cart problem without making it the required path.

**Watch for:** Keep `RUN_GENERATIVE_EXTENSION = False` for the baseline; if enabled, compare the lift against classical augmentation.

In [7]:
# Optional extension. Run the baseline first, then set this to True if your team wants the cGAN path.
RUN_GENERATIVE_EXTENSION = False
GENERATIVE_STEPS = 300
GENERATIVE_IMAGES_PER_CLASS = 8

import os, math, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image

gen_df = pd.DataFrame(columns=['path', 'fine'])

if RUN_GENERATIVE_EXTENSION:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    os.makedirs('generated_cgan', exist_ok=True)
    rare_to_i = {fine: i for i, fine in enumerate(rare)}
    rare_src = src[src.fine.isin(rare)].reset_index(drop=True)

    class RareCrops(Dataset):
        def __init__(self, table):
            self.table = table
            self.tf = T.Compose([T.ToImage(), T.Resize((64,64)), T.ToDtype(torch.float32, scale=True),
                                 T.Normalize([0.5]*3, [0.5]*3)])
        def __len__(self): return len(self.table)
        def __getitem__(self, i):
            row = self.table.iloc[i]
            x = self.tf(Image.open(row.path).convert('RGB'))
            y = torch.tensor(rare_to_i[row.fine]).long()
            return x, y

    class Generator(nn.Module):
        def __init__(self, z_dim, n_classes):
            super().__init__()
            self.label = nn.Embedding(n_classes, 16)
            self.fc = nn.Linear(z_dim + 16, 256 * 8 * 8)
            self.net = nn.Sequential(
                nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
                nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
                nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
        def forward(self, z, y):
            h = torch.cat([z, self.label(y)], dim=1)
            return self.net(self.fc(h).view(len(z), 256, 8, 8))

    class Discriminator(nn.Module):
        def __init__(self, n_classes):
            super().__init__()
            self.label = nn.Embedding(n_classes, 64 * 64)
            self.net = nn.Sequential(
                nn.Conv2d(4, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
                nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
                nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
                nn.Flatten(), nn.Linear(256 * 8 * 8, 1))
        def forward(self, x, y):
            label_map = self.label(y).view(len(y), 1, 64, 64)
            return self.net(torch.cat([x, label_map], dim=1)).squeeze(1)

    z_dim, n_classes = 64, len(rare)
    loader = DataLoader(RareCrops(rare_src), batch_size=16, shuffle=True, drop_last=True)
    G, D = Generator(z_dim, n_classes).to(device), Discriminator(n_classes).to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    data_iter = iter(loader)
    for step in range(GENERATIVE_STEPS):
        try:
            real, y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader); real, y = next(data_iter)
        real, y = real.to(device), y.to(device)
        z = torch.randn(len(real), z_dim, device=device)
        fake = G(z, y).detach()
        d_loss = F.binary_cross_entropy_with_logits(D(real, y), torch.ones(len(real), device=device)) + \
                 F.binary_cross_entropy_with_logits(D(fake, y), torch.zeros(len(real), device=device))
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()

        z = torch.randn(len(real), z_dim, device=device)
        fake = G(z, y)
        g_loss = F.binary_cross_entropy_with_logits(D(fake, y), torch.ones(len(real), device=device))
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
        if (step + 1) % 100 == 0:
            print('cGAN step', step + 1, 'D', round(float(d_loss), 3), 'G', round(float(g_loss), 3))

    rows = []
    G.eval()
    with torch.no_grad():
        for fine, class_id in rare_to_i.items():
            y = torch.full((GENERATIVE_IMAGES_PER_CLASS,), class_id, device=device, dtype=torch.long)
            z = torch.randn(GENERATIVE_IMAGES_PER_CLASS, z_dim, device=device)
            imgs = G(z, y)
            for k, img in enumerate(imgs):
                path = f'generated_cgan/{fine}_{k}.jpg'
                save_image((img + 1) / 2, path)
                rows.append({'path': path, 'fine': fine})
    gen_df = pd.DataFrame(rows)
    print('created', len(gen_df), 'cGAN/equivalent generated images')
else:
    print('cGAN/equivalent extension skipped; using classical augmentation only')


cGAN/equivalent extension skipped; using classical augmentation only


## Retrain head (v2)

**What:** Rebuild features over the original images plus the selected extra images and retrain the head.

**Why:** The v2 head sees a richer long tail; classical augmentation is required, generated images are optional.

**Watch for:** Keep the class ordering identical to Day 3 so before/after numbers are comparable.

In [8]:
import torch, torchvision.transforms.v2 as T
from PIL import Image
import numpy as np
import os
# Shared image preprocessing for DINOv2.
TF = T.Compose([T.ToImage(), T.Resize((224,224)), T.ToDtype(torch.float32, scale=True),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def feats_of(paths, model, bs=16):
    """Embed image files in small batches and return an (N, D) numpy array."""
    out=[]
    for i in range(0,len(paths),bs):
        xs=torch.stack([TF(Image.open(p).convert('RGB')) for p in paths[i:i+bs]])
        out.append(extract_features(model,[xs]))
    return np.concatenate(out,0)
from sklearn.model_selection import train_test_split
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)
model = load_backbone(device=device)
base = list_images(root)  # same clean basis as Day 3
classes = sorted(base.fine.unique().tolist()); cls_to_i = {c:i for i,c in enumerate(classes)}

# Pull in Day 2's real detector crops too, same as Day 3's crop-aware head. Without this,
# retraining here would silently throw away that fix - Day 5 always prefers whichever
# head we save today (head_v2.pt) over Day 3's head.pt, so if we rebuild from clean
# photos alone, the fridge-background shortcut Day 3 fixed would come right back.
try:
    crops = b.get_table('crops_manifest.csv')
    crops = crops[crops.crop_path.apply(os.path.exists)]
    crops = crops if len(crops) else None
except FileNotFoundError:
    crops = None
print('detector crops available:', 0 if crops is None else len(crops))

base_pool = base[['path', 'fine']]
if crops is not None:
    base_pool = pd.concat([base_pool, crops[['crop_path', 'fine']].rename(columns={'crop_path': 'path'})], ignore_index=True)
base_pool = base_pool[base_pool.fine.isin(classes)].reset_index(drop=True)
print('base_before training pool:', len(base_pool), '(', len(base), 'clean +', 0 if crops is None else len(crops), 'real crops )')

# Hold out a validation split first, then add augmented images to TRAIN only.
feats_base = feats_of(list(base_pool.path), model); yb = np.array([cls_to_i[f] for f in base_pool.fine])
Xtr,Xval,ytr,yval,itr,ival = train_test_split(feats_base, yb, np.arange(len(yb)),
                                               test_size=0.3, random_state=0, stratify=yb)
extra_df = pd.concat([aug_df, gen_df], ignore_index=True)
print('extra training images:', len(extra_df), '(classical:', len(aug_df), 'generated:', len(gen_df), ')')
fs = feats_of(list(extra_df.path), model); ys = np.array([cls_to_i[f] for f in extra_df.fine])
Xtr2 = np.concatenate([Xtr, fs]); ytr2 = np.concatenate([ytr, ys])
head_before = LinearHead(feats_base.shape[1], len(classes)).fit(Xtr, ytr, epochs=200)
head_v2 = LinearHead(feats_base.shape[1], len(classes)).fit(Xtr2, ytr2, epochs=200)
print('retrained head_v2 with selected long-tail data (crop-aware baseline preserved)')


using device: cuda


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

detector crops available: 211
base_before training pool: 5632 ( 5421 clean + 211 real crops )
extra training images: 24 (classical: 24 generated: 0 )
retrained head_v2 with selected long-tail data (crop-aware baseline preserved)


## Measure the lift

**What:** Compare before/after accuracy on the rare classes specifically and write `lift_table.csv`.

**Why:** Augmentation is only worth it if the targeted classes actually improve on held-out data.

**Watch for:** Lift can be flat on a small class subset - the pipeline and metric matter more than the number here.

In [9]:
pb = head_before.predict(Xval); pa = head_v2.predict(Xval)
def acc_for(fine, pred):
    ci = cls_to_i[fine]; mask = yval==ci
    return float((pred[mask]==ci).mean()) if mask.any() else float('nan')
lift = pd.DataFrame({'fine': rare,
                     'acc_before': [acc_for(f, pb) for f in rare],
                     'acc_after':  [acc_for(f, pa) for f in rare]})
print(lift.to_string(index=False))
print('mean lift on rare classes:', round(float((lift.acc_after-lift.acc_before).mean()), 3))


                    fine  acc_before  acc_after
    Alpro-Shelf-Soy-Milk    0.894737   0.894737
Alpro-Blueberry-Soyghurt    1.000000   1.000000
Yoggi-Strawberry-Yoghurt    0.863636   0.863636
mean lift on rare classes: 0.0


## Save + carry forward

**What:** Save head_v2 and the lift table into the bundle.

**Why:** Day 5 prefers `head_v2.pt` (this improved head) when exporting to ONNX.

**Watch for:** Confirm head_v2.pt is listed in the carry-forward print.

In [10]:
from pathlib import Path
save_head(head_v2, str(Path(b.root)/'head_v2.pt')); b._note('head_v2.pt')
b.put_table('lift_table.csv', lift)
save_bundle(b)
print('\u25b6 Carries forward to Day 5: head_v2.pt + lift_table.csv')


[bundle] saved -> /content/drive/MyDrive/snaic/week_4/Project/SmartCart
▶ Carries forward to Day 5: head_v2.pt + lift_table.csv
